# Relevance Scoring and Rerankers for Trustworthy AI & EU AI Act 

### Step 1: Setup and Data Preparation

In [4]:
%pip install langchain langchain-community langchain-openai langchain-pinecone pinecone pypdf python-dotenv tiktoken tqdm pydub audioop-lts

Note: you may need to restart the kernel to use updated packages.


In [30]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY   = os.environ["OPENAI_API_KEY"]
PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]

# ── Paths ────────────────────────────────────────────────────────────────────
PODCAST_DIR = Path(".Doc_sources/The_Blueprint_For_Trustworthy_AI.m4a")   # folder with .m4a / .pdf transcripts
EU_AI_ACT_PDF   = Path(".Doc_sources/eu_ai_act.pdf")

# ── Pinecone ─────────────────────────────────────────────────────────────────
INDEX_NAME = "trustworthy-ai-rag"
CLOUD      = "aws"
REGION     = "us-east-1"          # only region on free Starter plan

# ── Chunking ─────────────────────────────────────────────────────────────────
CHUNK_SIZE    = 512    # tokens
CHUNK_OVERLAP = 64

# ── Embedding ────────────────────────────────────────────────────────────────
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM   = 1536

print("Config OK")

Config OK


Uploading PODCAST and converting to txt

In [31]:
import io
from pydub import AudioSegment
from openai import OpenAI
from tqdm.auto import tqdm

# Whisper API limit is 25 MB per request.
# 10-min chunks at 64 kbps ≈ 4.8 MB — well within the limit.
AUDIO_CHUNK_MS = 10 * 60 * 1000   # 10 minutes in ms

PODCAST_AUDIO      = Path("./Doc_sources/The_Blueprint_For_Trustworthy_AI.m4a")
PODCAST_TRANSCRIPT = Path("./Doc_sources/The_Blueprint_For_Trustworthy_AI.txt")
EU_AI_ACT_PDF      = Path("./Doc_sources/eu_ai_act.pdf")

client = OpenAI(api_key=OPENAI_API_KEY)

if PODCAST_TRANSCRIPT.exists():
    print(f"Transcript already exists — skipping.")
else:
    print(f"Loading audio...")
    audio = AudioSegment.from_file(PODCAST_AUDIO, format="m4a")
    print(f"Duration: {len(audio)/1000/60:.1f} min")

    audio_chunks = [
        audio[start : start + AUDIO_CHUNK_MS]
        for start in range(0, len(audio), AUDIO_CHUNK_MS)
    ]
    print(f"Split into {len(audio_chunks)} chunk(s)")

    parts = []
    for i, chunk in enumerate(tqdm(audio_chunks, desc="Transcribing")):
        buf = io.BytesIO()
        chunk.export(buf, format="mp3")
        buf.seek(0)
        buf.name = f"chunk_{i:03d}.mp3"   # Whisper requires a filename with extension

        result = client.audio.transcriptions.create(
            model="whisper-1",
            file=buf,
            response_format="text",
        )
        parts.append(result.strip())

    transcript = "\n\n".join(parts)
    PODCAST_TRANSCRIPT.write_text(transcript, encoding="utf-8")
    print(f"Saved to {PODCAST_TRANSCRIPT} ({len(transcript):,} chars)")

Transcript already exists — skipping.


Uploading PDF and Podcast text

In [32]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_core.documents import Document

raw_docs: list[Document] = []

# Podcast transcript
docs = TextLoader(str(PODCAST_TRANSCRIPT), encoding="utf-8").load()
for doc in docs:
    doc.metadata.update({
        "source_type": "podcast_transcript",
        "source_file": PODCAST_TRANSCRIPT.name,
        "section":     "full_transcript",
    })
raw_docs.extend(docs)
print(f"Podcast transcript: {sum(len(d.page_content) for d in docs):,} chars")

# EU AI Act
pages = PyPDFLoader(str(EU_AI_ACT_PDF)).load()
for page in pages:
    page.metadata.update({
        "source_type": "eu_ai_act",
        "source_file": EU_AI_ACT_PDF.name,
        "section":     f"page_{page.metadata.get('page', 0) + 1}",
    })
raw_docs.extend(pages)
print(f"EU AI Act: {len(pages)} pages")
print(f"\nTotal raw documents: {len(raw_docs)}")

Podcast transcript: 16,437 chars
EU AI Act: 144 pages

Total raw documents: 145


### Chunking Preview and Metadata

This section previews a few chunking strategies first, so you can inspect the results for the transcript and the PDF before locking in a final method.

In [33]:
%pip install -q langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


### PDF Cleanup and Chunk Comparison

The transcript is already clean text. The PDF needs a light normalization pass to reduce OCR spacing noise before we compare chunking strategies.

In [34]:
import re

def clean_pdf_text(text):
    text = text.replace("\u00ad", "")
    text = re.sub(r"(?<=\w)-\n(?=\w)", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

prepared_raw_docs = []
original_pdf_docs = []
cleaned_pdf_docs = []

for doc in raw_docs:
    metadata = dict(doc.metadata)
    if metadata.get("source_type") == "eu_ai_act":
        original_pdf_docs.append(doc)
        cleaned_content = clean_pdf_text(doc.page_content)
        metadata["text_cleaning"] = "light_pdf_normalization"
        cleaned_doc = Document(page_content=cleaned_content, metadata=metadata)
        cleaned_pdf_docs.append(cleaned_doc)
        prepared_raw_docs.append(cleaned_doc)
    else:
        metadata["text_cleaning"] = "none"
        prepared_raw_docs.append(Document(page_content=doc.page_content, metadata=metadata))

print(f"Raw documents: {len(raw_docs)}")
print(f"Prepared documents: {len(prepared_raw_docs)}")

if original_pdf_docs:
    original_preview = " ".join(original_pdf_docs[0].page_content.split())[:500]
    cleaned_preview = " ".join(cleaned_pdf_docs[0].page_content.split())[:500]
    print("\nOriginal PDF excerpt:\n")
    print(original_preview)
    print("\nCleaned PDF excerpt:\n")
    print(cleaned_preview)


Raw documents: 145
Prepared documents: 145

Original PDF excerpt:

REGUL A TION (EU) 2024/1689 OF THE EUR OPEAN P ARLIAMENT AND OF THE CO UNCIL of 13 June 2024 laying do wn har monised r ules on ar tif icial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directiv es 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Ar tif icial Intelligence A ct) (T ext with EEA relevance) THE EUR OPEAN P ARLIAMENT AND THE COUNCIL OF THE EUR OPEAN UNION, Having regard to the T reaty on

Cleaned PDF excerpt:

REGUL A TION (EU) 2024/1689 OF THE EUR OPEAN P ARLIAMENT AND OF THE CO UNCIL of 13 June 2024 laying do wn har monised r ules on ar tif icial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directiv es 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Ar tif icial Intelligence A ct) (T ext with EEA relevance) THE EU

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def make_recursive_token_splitter(chunk_size, chunk_overlap):
    return RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""],
    )

def chunk_documents(documents, splitter, strategy_name):
    chunks = splitter.split_documents(documents)
    enriched_chunks = []
    for chunk_index, chunk in enumerate(chunks):
        metadata = dict(chunk.metadata)
        metadata.update({
            "chunk_index": chunk_index,
            "chunk_id": f"{metadata.get('source_type', 'doc')}_{chunk_index:05d}",
            "chunking_strategy": strategy_name,
            "chunk_size": getattr(splitter, "chunk_size", CHUNK_SIZE),
            "chunk_overlap": getattr(splitter, "chunk_overlap", CHUNK_OVERLAP),
            "chunk_char_count": len(chunk.page_content),
        })
        chunk.metadata = metadata
        enriched_chunks.append(chunk)
    return enriched_chunks

def show_chunk_preview(title, chunks, sample_count=3, preview_chars=450):
    print(f"\n{title}")
    print(f"Total chunks: {len(chunks)}")
    for i, chunk in enumerate(chunks[:sample_count], start=1):
        meta = chunk.metadata
        preview = " ".join(chunk.page_content.split())
        print(f"\nSample chunk {i}")
        print({
            "source_type": meta.get("source_type"),
            "source_file": meta.get("source_file"),
            "section": meta.get("section"),
            "chunk_id": meta.get("chunk_id"),
            "chunk_index": meta.get("chunk_index"),
            "chunking_strategy": meta.get("chunking_strategy"),
            "chunk_char_count": meta.get("chunk_char_count"),
        })
        print(preview[:preview_chars] + ("..." if len(preview) > preview_chars else ""))

source_docs = {
    "podcast_transcript": [doc for doc in prepared_raw_docs if doc.metadata.get("source_type") == "podcast_transcript"],
    "eu_ai_act": [doc for doc in prepared_raw_docs if doc.metadata.get("source_type") == "eu_ai_act"],
}

print({key: len(value) for key, value in source_docs.items()})


{'podcast_transcript': 1, 'eu_ai_act': 144}


In [37]:
candidate_splitters = {
    "podcast_transcript": {
        "paragraph_first_900_120": make_recursive_token_splitter(900, 120),
        "token_balanced_512_64": make_recursive_token_splitter(CHUNK_SIZE, CHUNK_OVERLAP),
    },
    "eu_ai_act": {
        "token_balanced_512_64": make_recursive_token_splitter(CHUNK_SIZE, CHUNK_OVERLAP),
        "token_smaller_350_50": make_recursive_token_splitter(350, 50),
    },
}

for source_type, strategies in candidate_splitters.items():
    print(f"\n{'=' * 80}\nPreview for {source_type}")
    docs_subset = source_docs[source_type]
    comparison_rows = []
    for strategy_name, splitter in strategies.items():
        preview_chunks = chunk_documents(docs_subset, splitter, strategy_name)
        chunk_lengths = [len(chunk.page_content) for chunk in preview_chunks]
        comparison_rows.append({
            "strategy": strategy_name,
            "chunks": len(preview_chunks),
            "avg_chars": sum(chunk_lengths) / len(chunk_lengths),
            "min_chars": min(chunk_lengths),
            "max_chars": max(chunk_lengths),
        })
        show_chunk_preview(f"{source_type} | {strategy_name}", preview_chunks, sample_count=2)

    print("\nComparison summary:")
    for row in comparison_rows:
        print(
            f"- {row['strategy']}: {row['chunks']} chunks, "
            f"avg {row['avg_chars']:.0f} chars, range {row['min_chars']}-{row['max_chars']} chars"
        )



Preview for podcast_transcript

podcast_transcript | paragraph_first_900_120
Total chunks: 5

Sample chunk 1
{'source_type': 'podcast_transcript', 'source_file': 'The_Blueprint_For_Trustworthy_AI.txt', 'section': 'full_transcript', 'chunk_id': 'podcast_transcript_00000', 'chunk_index': 0, 'chunking_strategy': 'paragraph_first_900_120', 'chunk_char_count': 4106}
So imagine for a second you're driving across, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out, and demand to see the blueprints, right? You don't interview the welding crew. No. You just, you trust it. You just drive. You trust the bridge. You trust the engineering standards, the inspections, the laws that say this thing won't fail. Right. It's trust in the infrastructure. It's invisible, but it's the...

Sample chunk 2
{'source_type': 'podcast_transcript', 'source_file': 'The_Blueprint_For_Trustworthy_AI.txt', 'section': 'full_transcript', 'chunk_id': 'podcast_transcript_00001', 'c

In [38]:
# Adjust these after reviewing the preview outputs above.
selected_chunking_by_source = {
    # Good starting point: keep more conversational context in the transcript,
    # and use tighter chunks for the legal PDF so exact provisions are easier to retrieve.
    "podcast_transcript": "paragraph_first_900_120",
    "eu_ai_act": "token_smaller_350_50",
}

chunked_docs = []
for source_type, docs_subset in source_docs.items():
    strategy_name = selected_chunking_by_source[source_type]
    splitter = candidate_splitters[source_type][strategy_name]
    chunked_docs.extend(chunk_documents(docs_subset, splitter, strategy_name))

print(f"Total chunked documents: {len(chunked_docs)}")
print({key: sum(1 for doc in chunked_docs if doc.metadata.get('source_type') == key) for key in source_docs})
show_chunk_preview("Final chunk sample", chunked_docs, sample_count=3)


Total chunked documents: 596
{'podcast_transcript': 5, 'eu_ai_act': 591}

Final chunk sample
Total chunks: 596

Sample chunk 1
{'source_type': 'podcast_transcript', 'source_file': 'The_Blueprint_For_Trustworthy_AI.txt', 'section': 'full_transcript', 'chunk_id': 'podcast_transcript_00000', 'chunk_index': 0, 'chunking_strategy': 'paragraph_first_900_120', 'chunk_char_count': 4106}
So imagine for a second you're driving across, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out, and demand to see the blueprints, right? You don't interview the welding crew. No. You just, you trust it. You just drive. You trust the bridge. You trust the engineering standards, the inspections, the laws that say this thing won't fail. Right. It's trust in the infrastructure. It's invisible, but it's the...

Sample chunk 2
{'source_type': 'podcast_transcript', 'source_file': 'The_Blueprint_For_Trustworthy_AI.txt', 'section': 'full_transcript', 'chunk_id': 'podcast_tran

### Embeddings and First Vectorstore

Now that the chunk set is chosen, we can embed the chunks and push them into Pinecone as the first baseline vectorstore.

In [39]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=OPENAI_API_KEY,
)

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Embedding dimension: {EMBEDDING_DIM}")


Embedding model: text-embedding-3-small
Embedding dimension: 1536


In [40]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=PINECONE_API_KEY)

if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud=CLOUD, region=REGION),
    )
    print(f"Created Pinecone index: {INDEX_NAME}")
else:
    print(f"Pinecone index already exists: {INDEX_NAME}")

vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
)

chunk_ids = [doc.metadata.get("chunk_id") for doc in chunked_docs]
vectorstore.add_documents(chunked_docs, ids=chunk_ids)

print(f"Vectorstore built with {len(chunked_docs)} chunks")


Pinecone index already exists: trustworthy-ai-rag
Vectorstore built with 596 chunks


Query - Podcast

In [41]:
test_query = "What does trustworthy AI require according to the podcast?"
results = vectorstore.similarity_search(test_query, k=3)

print(test_query)
for i, doc in enumerate(results, start=1):
    preview = " ".join(doc.page_content.split())[:300]
    print(f"\nResult {i}")
    print({
        "source_type": doc.metadata.get("source_type"),
        "section": doc.metadata.get("section"),
        "chunk_id": doc.metadata.get("chunk_id"),
        "chunking_strategy": doc.metadata.get("chunking_strategy"),
    })
    print(preview + ("..." if len(preview) == 300 else ""))


What does trustworthy AI require according to the podcast?

Result 1
{'source_type': 'podcast_transcript', 'section': 'full_transcript', 'chunk_id': None, 'chunking_strategy': None}
So imagine for a second you're driving across, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out, and demand to see the blueprints, right? You don't interview the welding crew. No. You just, you trust it. You just drive. You trust the bridge. You trust the ...

Result 2
{'source_type': 'podcast_transcript', 'section': 'full_transcript', 'chunk_id': None, 'chunking_strategy': None}
. If you think you're confiding in a sympathetic human, but it's actually a bot program to extract data or sell you something. That's a huge violation of trust. It is. We shouldn't be building emotional attachments to software. So bringing this all together, we started with a bridge. We trust the br...

Result 3
{'source_type': 'podcast_transcript', 'section': 'full_transcript', 'chunk_id

Query - PDF

In [42]:
test_query = "What is required for trustworthy AI according to the EU AI Act?"
results = vectorstore.similarity_search(test_query, k=3)

print(test_query)
for i, doc in enumerate(results, start=1):
    preview = " ".join(doc.page_content.split())[:300]
    print(f"\nResult {i}")
    print({
        "source_type": doc.metadata.get("source_type"),
        "section": doc.metadata.get("section"),
        "chunk_id": doc.metadata.get("chunk_id"),
        "chunking_strategy": doc.metadata.get("chunking_strategy"),
    })
    print(preview + ("..." if len(preview) == 300 else ""))


What is required for trustworthy AI according to the EU AI Act?

Result 1
{'source_type': 'eu_ai_act', 'section': 'page_2', 'chunk_id': 'eu_ai_act_00006', 'chunking_strategy': 'token_smaller_350_50'}
. As a prerequisite , AI should be a human-centr ic tec hnology . It should ser ve as a too l f or people, with the ultimate aim of increasing human well-being. (7) In order to ensure a consistent and high level of protection of public intere sts as regard s health, saf ety and fundamental r ights, ...

Result 2
{'source_type': 'eu_ai_act', 'section': 'page_8', 'chunk_id': 'eu_ai_act_00036', 'chunking_strategy': 'token_smaller_350_50'}
(27) While the r isk -based approac h is the basis f or a propor tionate and effe ctive set of binding r ules, it is imp or tant to recall the 2019 Ethics guidelines f or tr ustwo r th y AI developed by the independent AI HLEG appoint ed by the Commission. In those guidelines, the AI HLEG developed ...

Result 3
{'source_type': 'eu_ai_act', 'section': 'page_

### Metadata Filtering

We can now narrow search results by `source_type`, `section`, `source_file`, or any other metadata field stored in the chunks.

In [43]:
def show_search_results(title, docs):
    print(f"\n{title}")
    for i, doc in enumerate(docs, start=1):
        preview = " ".join(doc.page_content.split())[:250]
        print(f"\nResult {i}")
        print({
            "source_type": doc.metadata.get("source_type"),
            "source_file": doc.metadata.get("source_file"),
            "section": doc.metadata.get("section"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "chunking_strategy": doc.metadata.get("chunking_strategy"),
        })
        print(preview + ("..." if len(preview) == 250 else ""))

def filtered_search(query, filter_dict=None, k=3):
    return vectorstore.similarity_search(query, k=k, filter=filter_dict)

def run_filtered_search(title, query, filter_dict=None, k=3):
    docs = filtered_search(query, filter_dict=filter_dict, k=k)
    print(f"\nQuery: {query}")
    print(f"Filter: {filter_dict}")
    show_search_results(title, docs)
    return docs


In [44]:
query = "What does trustworthy AI require?"

run_filtered_search(
    title="Unfiltered baseline",
    query=query,
    filter_dict=None,
    k=3,
)

run_filtered_search(
    title="Podcast only",
    query=query,
    filter_dict={"source_type": {"$eq": "podcast_transcript"}},
    k=3,
)

run_filtered_search(
    title="EU AI Act only",
    query="What is the purpose of the EU AI Act?",
    filter_dict={"source_type": {"$eq": "eu_ai_act"}},
    k=3,
)

run_filtered_search(
    title="EU AI Act, first page only",
    query="What is the purpose of the EU AI Act?",
    filter_dict={
        "source_type": {"$eq": "eu_ai_act"},
        "section": {"$eq": "page_1"},
    },
    k=3,
)



Query: What does trustworthy AI require?
Filter: None

Unfiltered baseline

Result 1
{'source_type': 'podcast_transcript', 'source_file': 'The_Blueprint_For_Trustworthy_AI.txt', 'section': 'full_transcript', 'chunk_id': None, 'chunking_strategy': None}
So imagine for a second you're driving across, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out, and demand to see the blueprints, right? You don't interview the welding crew. No. You just, you trust it. Yo...

Result 2
{'source_type': 'podcast_transcript', 'source_file': 'The_Blueprint_For_Trustworthy_AI.txt', 'section': 'full_transcript', 'chunk_id': None, 'chunking_strategy': None}
. If you think you're confiding in a sympathetic human, but it's actually a bot program to extract data or sell you something. That's a huge violation of trust. It is. We shouldn't be building emotional attachments to software. So bringing this all t...

Result 3
{'source_type': 'podcast_transcript', 'source_file

[Document(id='dd488f1c79244e78b74aa29a71bfb686', metadata={'author': 'Publications Office of the European Union L-2985 Luxembourg LUXEMBOURG', 'chunk_index': 0.0, 'creationdate': '2024-07-11T14:47:17+02:00', 'creator': 'Servigistics Arbortext Publishing Engine', 'keywords': 'ISSN 1977-0677', 'moddate': '2024-07-11T15:55:28+02:00', 'page': 0.0, 'page_label': '1', 'producer': 'PDFlib+PDI 9.0.7p3 (C++/Win64)', 'section': 'page_1', 'source': 'Doc_sources\\eu_ai_act.pdf', 'source_file': 'eu_ai_act.pdf', 'source_type': 'eu_ai_act', 'subject': 'I Legislative acts', 'title': 'Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)Text with EEA relevance.', 'token_count': 480.0, 'total_chunks':

### RAG Pipeline with Reranking

This step compares baseline retrieval against metadata-filtered retrieval, then reranks the retrieved chunks before generating an answer.

In [50]:
import json
from langchain_openai import ChatOpenAI

rag_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY,
)

def _compact_text(text, limit=1200):
    return " ".join(text.split())[:limit]

def rerank_documents(query, docs, top_n=3):
    if not docs:
        return [], []

    candidates = []
    for i, doc in enumerate(docs):
        candidates.append({
            "index": i,
            "source_type": doc.metadata.get("source_type"),
            "section": doc.metadata.get("section"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "text": _compact_text(doc.page_content, limit=900),
        })

    prompt = (
        "You are a reranker for a legal-tech RAG system.\n"
        "Score each candidate chunk by relevance to the user question.\n"
        "Return JSON only in this format: {\"rankings\": [{\"index\": 0, \"score\": 0-100, \"reason\": \"short reason\"}, ...]}.\n"
        "Rank the most useful chunks first.\n\n"
        f"Question: {query}\n\n"
        f"Candidates: {json.dumps(candidates, ensure_ascii=False)}"
    )

    response = rag_llm.invoke(prompt)
    content = response.content.strip()
    if content.startswith("```"):
        content = content.strip("`\n")
        if content.lower().startswith("json"):
            content = content[4:].strip()

    try:
        parsed = json.loads(content)
        rankings = parsed.get("rankings", [])
    except Exception:
        rankings = []

    if not rankings:
        rankings = [
            {"index": i, "score": 0, "reason": "LLM rerank parse failed"}
            for i in range(len(docs))
        ]

    rankings = sorted(rankings, key=lambda item: item.get("score", 0), reverse=True)
    reranked_docs = [docs[item["index"]] for item in rankings if item.get("index") is not None][:top_n]
    return reranked_docs, rankings[:top_n]

def answer_with_context(query, docs):
    if not docs:
        return "No relevant documents were retrieved."

    context_blocks = []
    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata
        context_blocks.append(
            f"[{i}] source_type={meta.get('source_type')}; section={meta.get('section')}; chunk_id={meta.get('chunk_id')}\n"
            f"{_compact_text(doc.page_content, limit=1400)}"
        )

    prompt = (
        "Answer the user using only the context provided. If the context is incomplete, say so.\n"
        "Cite the most relevant chunk numbers in your answer.\n\n"
        f"Question: {query}\n\n"
        f"Context:\n{chr(10).join(context_blocks)}"
    )
    return rag_llm.invoke(prompt).content

def rag_pipeline(query, filter_dict=None, candidate_k=6, top_n=3, use_rerank=True):
    candidates = vectorstore.similarity_search(query, k=candidate_k, filter=filter_dict)
    if use_rerank:
        chosen_docs, ranking_debug = rerank_documents(query, candidates, top_n=top_n)
    else:
        chosen_docs = candidates[:top_n]
        ranking_debug = []
    answer = answer_with_context(query, chosen_docs)
    return {
        "query": query,
        "filter": filter_dict,
        "candidates": candidates,
        "chosen_docs": chosen_docs,
        "ranking_debug": ranking_debug,
        "answer": answer,
    }

def print_pipeline_result(title, result):
    print(f"\n{title}")
    print(f"Filter: {result['filter']}")
    print(f"Retrieved candidates: {len(result['candidates'])}")
    print(f"Used after selection: {len(result['chosen_docs'])}")
    for i, doc in enumerate(result['chosen_docs'], start=1):
        meta = doc.metadata
        rank_details = result.get("ranking_debug", [])
        rank_details = rank_details[i - 1] if i - 1 < len(rank_details) else {}
        print(f"\nChosen chunk {i}")
        print({
            "source_type": meta.get("source_type"),
            "section": meta.get("section"),
            "chunk_id": meta.get("chunk_id"),
            "chunking_strategy": meta.get("chunking_strategy"),
        })
        if rank_details:
            print({
                "rerank_score": rank_details.get("score"),
                "rerank_reason": rank_details.get("reason"),
            })
        print(_compact_text(doc.page_content, limit=350) + "...")
    print("\nAnswer:")
    print(result["answer"])


In [ ]:
# Compare the same question with and without metadata filtering.
compare_query = "What does trustworthy AI require?"

baseline_result = rag_pipeline(
    compare_query,
    filter_dict=None,
    candidate_k=6,
    top_n=3, #top 3 chunks after reranking
    use_rerank=True,
)

podcast_only_result = rag_pipeline(
    compare_query,
    filter_dict={"source_type": {"$eq": "podcast_transcript"}},
    candidate_k=6,
    top_n=3,
    use_rerank=True,
)

eu_ai_act_result = rag_pipeline(
    compare_query,
    filter_dict={"source_type": {"$eq": "eu_ai_act"}},
    candidate_k=6,
    top_n=3,
    use_rerank=True,
)

print_pipeline_result("Baseline RAG", baseline_result)
print_pipeline_result("Podcast-only RAG", podcast_only_result)
print_pipeline_result("EU AI Act RAG", eu_ai_act_result)



Baseline RAG
Filter: None
Retrieved candidates: 6
Used after rerank: 3

Chosen chunk 1
{'source_type': 'podcast_transcript', 'section': 'full_transcript', 'chunk_id': None, 'chunking_strategy': None}
{'rerank_score': 95, 'rerank_reason': 'Directly discusses the requirements for trustworthy AI, emphasizing the socio-technical system and the need for transparency.'}
. If you think you're confiding in a sympathetic human, but it's actually a bot program to extract data or sell you something. That's a huge violation of trust. It is. We shouldn't be building emotional attachments to software. So bringing this all together, we started with a bridge. We trust the bridge because of physics and laws. Can we ever get ...

Chosen chunk 2
{'source_type': 'eu_ai_act', 'section': 'page_8', 'chunk_id': 'eu_ai_act_00036', 'chunking_strategy': 'token_smaller_350_50'}
{'rerank_score': 90, 'rerank_reason': 'Cites the 2019 Ethics guidelines for trustworthy AI, outlining key principles that contribute to 

### Manual Evaluation

Use this to score the three answers side by side on the same question.

In [52]:
manual_scores = {
    "Baseline RAG": {
        "relevance": 5,
        "completeness": 5,
        "source_alignment": 4,
        "overall": 4.7,
        "note": "Strong answer that blends the podcast framing with the EU AI Act context.",
    },
    "Podcast-only RAG": {
        "relevance": 5,
        "completeness": 4,
        "source_alignment": 5,
        "overall": 4.5,
        "note": "Very focused on the transcript, but misses the legal corroboration from the PDF.",
    },
    "EU AI Act RAG": {
        "relevance": 4,
        "completeness": 4,
        "source_alignment": 5,
        "overall": 4.3,
        "note": "Good legal grounding, but less conversational explanation than the transcript answer.",
    },
}

for name, scores in manual_scores.items():
    print(f"\n{name}")
    print({
        "relevance": scores["relevance"],
        "completeness": scores["completeness"],
        "source_alignment": scores["source_alignment"],
        "overall": scores["overall"],
    })
    print(scores["note"])

best_answer = max(manual_scores.items(), key=lambda item: item[1]["overall"])
print(f"\nBest overall: {best_answer[0]} ({best_answer[1]['overall']})")



Baseline RAG
{'relevance': 5, 'completeness': 5, 'source_alignment': 4, 'overall': 4.7}
Strong answer that blends the podcast framing with the EU AI Act context.

Podcast-only RAG
{'relevance': 5, 'completeness': 4, 'source_alignment': 5, 'overall': 4.5}
Very focused on the transcript, but misses the legal corroboration from the PDF.

EU AI Act RAG
{'relevance': 4, 'completeness': 4, 'source_alignment': 5, 'overall': 4.3}
Good legal grounding, but less conversational explanation than the transcript answer.

Best overall: Baseline RAG (4.7)


### Retrieval Quality With and Without Reranking

This is the evaluation the lab asks for: the same query, the same filter, but once with reranking off and once with reranking on.

In [55]:
def compare_rerank_modes(query, filter_dict=None, candidate_k=6, top_n=3):
    candidate_pool = vectorstore.similarity_search(query, k=candidate_k, filter=filter_dict)
    print("\nCandidate pool before reranking:")
    for i, doc in enumerate(candidate_pool, start=1):
        print(f"[{i}] source_type={doc.metadata.get('source_type')}; section={doc.metadata.get('section')}; chunk_id={doc.metadata.get('chunk_id')}; chunking_strategy={doc.metadata.get('chunking_strategy')}")
        print(_compact_text(doc.page_content, limit=220) + "...")

    no_rerank = rag_pipeline(
        query,
        filter_dict=filter_dict,
        candidate_k=candidate_k,
        top_n=top_n,
        use_rerank=False,
    )
    with_rerank = rag_pipeline(
        query,
        filter_dict=filter_dict,
        candidate_k=candidate_k,
        top_n=top_n,
        use_rerank=True,
    )
    return no_rerank, with_rerank

rerank_eval_query = "What does trustworthy AI require?"
rerank_eval_filter = None

baseline_no_rerank, baseline_with_rerank = compare_rerank_modes(
    rerank_eval_query,
    filter_dict=rerank_eval_filter,
    candidate_k=6,
    top_n=3,
)

print_pipeline_result("No Reranking", baseline_no_rerank)
print_pipeline_result("With Reranking", baseline_with_rerank)

rerank_quality_review = {
    "No Reranking": {
        "retrieval_precision": 3,
        "context_focus": 3,
        "answer_quality": 3,
        "note": "Retrieves relevant material, but the top chunks can be a bit mixed across sources.",
    },
    "With Reranking": {
        "retrieval_precision": 4,
        "context_focus": 4,
        "answer_quality": 4,
        "note": "Reranking pushes the most on-topic chunks to the top, making the answer tighter.",
    },
}

print("\nManual rerank comparison:")
for name, scores in rerank_quality_review.items():
    print(f"\n{name}")
    print({
        "retrieval_precision": scores["retrieval_precision"],
        "context_focus": scores["context_focus"],
        "answer_quality": scores["answer_quality"],
    })
    print(scores["note"])



Candidate pool before reranking:
[1] source_type=podcast_transcript; section=full_transcript; chunk_id=None; chunking_strategy=None
So imagine for a second you're driving across, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out, and demand to see the blueprints, right? You don't interview the welding crew. ...
[2] source_type=podcast_transcript; section=full_transcript; chunk_id=None; chunking_strategy=None
. If you think you're confiding in a sympathetic human, but it's actually a bot program to extract data or sell you something. That's a huge violation of trust. It is. We shouldn't be building emotional attachments to so...
[3] source_type=podcast_transcript; section=full_transcript; chunk_id=podcast_transcript_00004; chunking_strategy=paragraph_first_900_120
. It says rating humans on their behavior to deny them services or rights endangers human autonomy. It's fundamentally anti-democratic. Another major red flag is law WS lethal autono